# Galileo Brain v2 — Fine-Tuning Qwen3-4B con 20K esempi

**10 celle**. Copia-incolla una per una su Google Colab. Dopo la cella 1, riavvia il runtime.

## Differenze vs PoC (500 esempi)
| | PoC (S75) | v2 (S117) |
|---|---|---|
| Dataset | 500 esempi | **20,000 esempi** |
| Epoche | 3 | **1** (piu' dati = meno epoche) |
| Multi-turn | no | **si** (2-4 turni, 2655 entries) |
| Intent coperti | 5 | **6** (+vision) |
| Action tag | 15 | **28** |
| Typo/slang | basico | **40% con errori realistici** |
| Tempo training | ~57 min (T4) | **~12h (T4) / ~3h (A100)** |
| max_seq_length | 2048 | **4096** (multi-turn piu' lunghi) |

## Setup
1. Install pinnato: `transformers==4.56.2` + `trl==0.22.2`
2. Modello: `unsloth/Qwen3-4B-Instruct-2507` (text-only pre-quantizzato)
3. `train_on_responses_only` — il modello impara SOLO le risposte, non il system prompt
4. **Consigliato**: Colab Pro con A100 per training in ~3h invece di ~12h

In [ ]:
# ============================================================
# CELLA 1 — Install (ESATTA da notebook ufficiale Unsloth)
# ============================================================
# DOPO QUESTA CELLA: Runtime > Restart session
# ============================================================

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {
        '2.10': '0.0.34',
        '2.9': '0.0.33.post1',
        '2.8': '0.0.32.post2'
    }.get(v, '0.0.34')
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

print("--- Install completato. Riavvia il runtime (Runtime > Restart session) ---")

In [ ]:
# ============================================================
# CELLA 2 — Carica modello + LoRA
# ============================================================
# CHECKPOINT: deve stampare Trainable > 0
# max_seq_length=4096 per multi-turn (entries con 5-9 messaggi)
# ============================================================

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=4096,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
assert trainable > 0, "ERRORE: 0 parametri trainabili!"
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"max_seq_length: 4096")

In [ ]:
# ============================================================
# CELLA 3 — Upload dataset (20K — ~80MB)
# ============================================================
# Upload galileo-brain-20k.jsonl dal tuo computer
# Generato con: python3 scripts/generate-brain-20k.py --seed 42
# ============================================================

from google.colab import files
import os, json

DATASET_PATH = "galileo-brain-20k.jsonl"
if not os.path.exists(DATASET_PATH):
    print("Carica galileo-brain-20k.jsonl (~80MB)...")
    print("(potrebbe servire 1-2 minuti per l'upload)")
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

# Quick stats
with open(DATASET_PATH) as f:
    lines = f.readlines()
num_lines = len(lines)
multi_turn = sum(1 for l in lines if l.count('"role"') > 3)
size_mb = os.path.getsize(DATASET_PATH) / 1e6

print(f"\nDataset: {DATASET_PATH}")
print(f"  Esempi: {num_lines:,}")
print(f"  Multi-turn: {multi_turn:,}")
print(f"  Dimensione: {size_mb:.1f} MB")

# Sanity check
first = json.loads(lines[0])
assert "messages" in first, "ERRORE: formato invalido!"
print(f"  Formato: OK (ChatML con {len(first['messages'])} messaggi nel primo esempio)")

In [ ]:
# ============================================================
# CELLA 4 — Prepara dataset con chat template
# ============================================================

from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

dataset = load_dataset("json", data_files={"train": DATASET_PATH}, split="train")
print(f"Colonne: {dataset.column_names}, Righe: {len(dataset)}")

def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"\nEsempio formattato (500 chars):\n{dataset[0]['text'][:500]}")

In [ ]:
# ============================================================
# CELLA 5 — SFTTrainer + train_on_responses_only
# ============================================================
# HYPERPARAMS SCALATI PER 20K:
# - epochs: 3 → 1  (40x piu' dati, 1 epoca basta)
# - warmup: 10 → 100  (proporzionale)
# - grad_accum: 4 → 8  (effective batch 16)
# - save_steps: 50 → 500  (checkpoint ogni ~20%)
# - logging: 5 → 25  (meno rumore)
# 
# Steps stimati: 20000 / (2 * 8) = 1250 steps
# Tempo stimato: ~6h (T4) / ~1.5h (A100)
# ============================================================

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=100,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="none",
        output_dir="galileo-brain-output",
        save_steps=500,
        save_total_limit=3,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Masking: loss calcolata SOLO sulle risposte assistant
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Stima tempo
total_steps = len(trainer.get_train_dataloader())
print(f"Trainer pronto. Loss solo su risposte assistant.")
print(f"Steps totali: {total_steps}")
print(f"Tempo stimato: ~{total_steps * 18 / 3600:.1f}h (T4) / ~{total_steps * 5 / 3600:.1f}h (A100)")

In [ ]:
# ============================================================
# CELLA 6 — Training
# ============================================================
# CHECKPOINT: loss finale deve scendere sotto ~0.3
# (con 20K il modello converge meglio del PoC)
# ============================================================

import time

start = time.time()
stats = trainer.train()
elapsed = time.time() - start

print(f"\n{'='*50}")
print(f"TRAINING COMPLETATO!")
print(f"Tempo: {elapsed/60:.1f} min ({elapsed/3600:.1f} ore)")
print(f"Loss finale: {stats.training_loss:.4f}")
print(f"GPU peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CELLA 7 — Test inferenza (6 intent + multi-turn + typo)
# ============================================================
# CHECKPOINT: almeno 8/10 devono produrre JSON valido
# ============================================================

import json
FastLanguageModel.for_inference(model)

# System prompt dal dataset
with open(DATASET_PATH) as f:
    first = json.loads(f.readline())
SYSTEM = first["messages"][0]["content"]

def test(msg, label=""):
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": msg}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs, max_new_tokens=512,
        temperature=0.1, top_p=0.95,
    )
    resp = tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    try:
        parsed = json.loads(resp)
        intent = parsed.get("intent", "?")
        actions = len(parsed.get("actions", []))
        nlm = parsed.get("needs_llm", "?")
        ok = True
    except:
        parsed = {"raw": resp[:200]}
        intent = "?"
        actions = 0
        nlm = "?"
        ok = False
    status = "\u2705" if ok else "\u26a0\ufe0f"
    print(f"  {status} [{label}] intent={intent} actions={actions} nlm={nlm}")
    if not ok:
        print(f"     RAW: {resp[:150]}")
    return ok

print("=== TEST INFERENZA (10 test, 6 intent) ===\n")

passed = 0
total = 10

# T1: action — play
passed += test("tab: simulator\ncomponenti: [led1, resistor1]\nfili: 2\n\n[MESSAGGIO]\nAvvia la simulazione", "action:play")

# T2: action — clearall con typo
passed += test("tab: simulator\ncomponenti: [led1]\nfili: 1\n\n[MESSAGGIO]\npulisci tuto", "action:clearall+typo")

# T3: circuit — place
passed += test("tab: simulator\ncomponenti: []\nfili: 0\n\n[MESSAGGIO]\nMetti un LED rosso sulla breadboard", "circuit:place")

# T4: circuit — multi-component
passed += test("tab: simulator\ncomponenti: []\nfili: 0\n\n[MESSAGGIO]\nCostruisci un circuito con LED e resistenza", "circuit:multi")

# T5: tutor — theory
passed += test("tab: simulator\ncomponenti: [resistor1]\nfili: 0\n\n[MESSAGGIO]\nChe cos'e' la legge di Ohm?", "tutor:theory")

# T6: code — question
passed += test("tab: editor\ncomponenti: [led1]\nfili: 1\nesperimento: v3-cap1-esp1\n\n[MESSAGGIO]\nCome faccio lampeggiare il LED con Arduino?", "code:blink")

# T7: navigation — load experiment
passed += test("tab: simulator\ncomponenti: []\nfili: 0\n\n[MESSAGGIO]\nCarica l'esperimento del semaforo", "nav:loadexp")

# T8: navigation — tab switch
passed += test("tab: simulator\ncomponenti: [led1]\nfili: 1\n\n[MESSAGGIO]\nApri il tab video", "nav:tab")

# T9: vision — analyze
passed += test("tab: simulator\ncomponenti: [led1, resistor1]\nfili: 3\n\n[MESSAGGIO]\nGuarda il mio circuito e dimmi se va bene", "vision:analyze")

# T10: robustness — kid slang
passed += test("tab: simulator\ncomponenti: []\nfili: 0\n\n[MESSAGGIO]\ndai metti la lucina xke voglio fare l'esperiment", "robust:slang")

print(f"\n{'='*50}")
print(f"RISULTATO: {passed}/{total} PASS")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CELLA 8 — Salva LoRA + GGUF
# ============================================================
# LoRA adapter per uso diretto con Transformers
# GGUF q4_k_m per Ollama/llama.cpp (~2.5 GB)
# ============================================================

# LoRA adapter
model.save_pretrained("galileo-brain-v2-lora")
tokenizer.save_pretrained("galileo-brain-v2-lora")
print("LoRA salvato in galileo-brain-v2-lora/")

# GGUF per Ollama
print("Conversione GGUF q4_k_m (potrebbe richiedere 5-10 min)...")
model.save_pretrained_gguf(
    "galileo-brain-v2-gguf", tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF q4_k_m salvato in galileo-brain-v2-gguf/")

In [ ]:
# ============================================================
# CELLA 9 — Download GGUF
# ============================================================

from google.colab import files
import os

gguf_dir = "galileo-brain-v2-gguf"
for f in os.listdir(gguf_dir):
    if f.endswith(".gguf"):
        size = os.path.getsize(os.path.join(gguf_dir, f)) / 1e9
        print(f"Scaricando {f} ({size:.2f} GB)...")
        files.download(os.path.join(gguf_dir, f))

print("\nFatto! Per usare con Ollama:")
print("  ollama create galileo-brain-v2 -f Modelfile")
print("  ollama run galileo-brain-v2")

In [ ]:
# ============================================================
# CELLA 10 (OPZIONALE) — Upload su HuggingFace Hub
# ============================================================
# Decommentare e inserire il tuo token HF per pushare il modello
# ============================================================

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
#
# model.push_to_hub_gguf(
#     "YOUR_USERNAME/galileo-brain-v2-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# print("Pushato su HuggingFace Hub!")

print("Per pushare su HF, decommenta il codice sopra e inserisci il tuo token.")